# Pickle and Joblib

---

In this notebook, we will learn how to save (serialize) and load (deserialize) trained machine learning models using Python's two primary tools: **pickle** and **joblib**.

We will cover:
- Saving and loading a model with `pickle`
- Saving and loading a model with `joblib` (and why it's preferred for ML)
- Comparing file sizes with and without compression
- The critical practice of persisting **entire Pipelines**, not just bare models
- Saving model metadata alongside the artifact
- Verifying that a loaded model produces identical predictions

---

## 1. Setup
Let's start by importing the necessary libraries and training a simple model that we can practice saving and loading.

In [1]:
import pickle
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [3]:
# Create a directory to save our model artifacts
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

In [4]:
# Load the Iris dataset
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size:     {X_test.shape[0]}")
print(f"Features:          {feature_names}")
print(f"Classes:           {target_names}")

Training set size: 120
Test set size:     30
Features:          ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes:           ['setosa' 'versicolor' 'virginica']



---

## 2. Pickle: Python's Built-in Serializer

`pickle` is Python's built-in module for serializing and deserializing any Python object into a byte stream.

#### Key characteristics:
- Part of the Python standard library (no installation needed)
- Can serialize virtually any Python object
- Uses `dump()` to write and `load()` to read
- Files are typically saved with a `.pkl` extension

> ⚠️ **Security Warning:** Pickle files can execute arbitrary code when loaded. **Never unpickle data from an untrusted source**. A malicious `.pkl` file can run harmful code the moment you call `pickle.load()`. Only load pickle files that you or your team have created.

Let's train a bare `RandomForestClassifier` and save it with pickle.

In [5]:
# Train a bare model (no pipeline, just the estimator)
# We'll need to manually scale the data first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_original = rf_model.predict(X_test_scaled)
original_accuracy = accuracy_score(y_test, y_pred_original)
print(f"Original model accuracy: {original_accuracy:.4f}")


Original model accuracy: 1.0000


In [6]:
# Save the model with pickle
pickle_path = MODELS_DIR / "rf_model_pickle.pkl"

with open(pickle_path, "wb") as f:  # "wb" = write bytes
    pickle.dump(rf_model, f)

print(f"Model saved to: {pickle_path}")
print(f"File size:      {pickle_path.stat().st_size / 1024:.1f} KB")

Model saved to: models/rf_model_pickle.pkl
File size:      173.1 KB


In [7]:
# Load the model back from disk
with open(pickle_path, "rb") as f:  # "rb" = read bytes
    rf_model_loaded = pickle.load(f)
    
# Verify: predictions must be identical
y_pred_loaded = rf_model_loaded.predict(X_test_scaled)
print(f"Predictions match: {np.array_equal(y_pred_original, y_pred_loaded)}")

Predictions match: True



---

## 3. Joblib: The ML-Optimized Alternative

`joblib is the serialization library **recommended by scikit-learn** for persisting models. It provides two key advantages over pickle:
1. **Efficiency with NumPy arrays:** ML models often contain large NumPy arrays (e.g., the learned weights of a Random Forest's many decision trees). Joblib is specifically optimized to serialize these efficiently.
2. **Built-in compression:** Joblib natively supports compressed saving, which can significantly reduce file sizes. This is useful when deploying models to cloud services or sending them to clients.

The API is almost identical to pickle: `joblib.dump()` and `joblib.load()`

In [8]:
# Save with joblib (no compression)
joblib_path = MODELS_DIR / "rf_model_joblib.joblib"
joblib.dump(rf_model, joblib_path)

print(f"Joblib (no compression): {joblib_path.stat().st_size / 1024:.1f} KB")

Joblib (no compression): 182.5 KB


In [9]:
# Save with joblib (compressed)
# compress=3 is a good balance between speed and size (range: 0-9)
joblib_compressed_path = MODELS_DIR / "rf_model_joblib_compressed.joblib"
joblib.dump(rf_model, joblib_compressed_path, compress=3)

print(f"Joblib (compressed=3):   {joblib_compressed_path.stat().st_size / 1024:.1f} KB")

Joblib (compressed=3):   25.7 KB


In [10]:
# Load and verify
rf_model_from_joblib = joblib.load(joblib_path)
y_pred_joblib = rf_model_from_joblib.predict(X_test_scaled)

print(f"Predictions match: {np.array_equal(y_pred_original, y_pred_joblib)}")

Predictions match: True



### 3.1. File Size Comparison
Let's compare the file sized of the three approaches side by side.

In [11]:
print("File Size Comparison")
print("=" * 45)
print(f"{'Method':<30} {'Size (KB)':>10}")
print("-" * 45)
print(f"{'Pickle (.pkl)':<30} {pickle_path.stat().st_size / 1024:>10.1f}")
print(f"{'Joblib (.joblib)':<30} {joblib_path.stat().st_size / 1024:>10.1f}")
print(f"{'Joblib (compressed=3)':<30} {joblib_compressed_path.stat().st_size / 1024:>10.1f}")

File Size Comparison
Method                          Size (KB)
---------------------------------------------
Pickle (.pkl)                       173.1
Joblib (.joblib)                    182.5
Joblib (compressed=3)                25.7


### 3.2. When to Use Which?

| | Pickle | Joblib |
| :--- | :--- | :--- |
| **Best for** | General Python objects (dicts, lists, configs) | ML models with large NumPy arrays |
| **Compression** | Not built-in (requires `gzip` wrapper) | Built-in (`compress` parameter) |
| **Speed** | Slightly slower with large arrays | Optimized for large arrays |
| **Recommendation** | Use for non-ML Python objects | **Use for scikit-learn models** |

**Rule of thumb:** If it's a trained ML model, use `joblib`. If it's a plain Python dictionary or config object, `pickle` is fine.

---

## 4. The Pipeline Rule: Save the Whole Pipeline

In the previous sections, 